# 01 - Data Audit
---
## Contexto

Este notebook tem como objetivo realizar a auditoria técnica da base de dados do e-commerce (2023–2025), identificando inconsistências estruturais, problemas de qualidade e potenciais riscos analíticos antes da etapa de análise exploratória.

A base contém ruídos corporativos realistas, simulando cenários comuns em ambientes transacionais reais.

---

## Objetivos da Auditoria

Avaliar:

- Estrutura geral da base
- Tipos de dados
- Valores ausentes
- Registros duplicados
- Inconsistências categóricas
- Anomalias numéricas
- Integridade temporal

A auditoria é fundamental para garantir que análises estratégicas posteriores sejam confiáveis.

## 📊 Data Dictionary

| Variável               | Tipo Esperado | Descrição |
|------------------------|--------------|-----------|
| ID da Venda            | Inteiro / ID | Identificador único da transação |
| Data da Venda          | Data         | Data em que a venda foi realizada |
| Produto                | Texto        | Nome do produto vendido |
| Categoria              | Texto        | Categoria à qual o produto pertence |
| Preço Unitário         | Float        | Valor unitário do produto antes de desconto |
| Quantidade Vendida     | Inteiro      | Número de unidades vendidas na transação |
| Desconto               | Float (%) ou valor monetário | Desconto aplicado na venda |
| Valor Total            | Float        | Valor final da venda após desconto |
| Localização            | Texto        | Cidade ou região associada à venda |
| Forma de Pagamento     | Texto        | Meio de pagamento utilizado pelo cliente |

### Pontos de Atenção Esperados

- "Data da Venda" deve estar dentro do intervalo 2023–2025.
- "Quantidade Vendida" não deve conter valores negativos ou zero.
- "Preço Unitário" e "Valor Total" não devem ser negativos.
- "Valor Total" deve ser consistente com:
  
- Valor Total ≈ (Preço Unitário × Quantidade Vendida) − Desconto

- "Categoria" e "Forma de Pagamento" devem estar padronizadas.

---
# 02 - Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/raw/vendas_loja_expandida.csv")
df.head()

,ID da Venda,Data da Venda,Produto,Categoria,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Localização,Forma de Pagamento
0,1001,2023-01-01 00:00:00.000000000,Livro G,Livros,60.00,6,0.05,57.00,Curitiba,Pix
1,1002,2023-01-01 05:49:40.386924616,Camisa D,Roupas,80.00,4,0.04,307.20,Salvador,Cartão de Débito
2,1003,2023-01-01 11:39:20.773849232,Calça E,Roupas,120.00,6,0.02,705.60,Curitiba,Boleto
3,1004,2023-01-01 17:29:01.160773849,Livro G,Livros,60.00,2,0.02,117.60,Salvador,Pix
4,1005,2023-01-01 23:18:41.547698465,Fone C,Eletrônicos,1626.54,2,0.07,3025.36,São Paulo,Cartão de Débito


---
# 03 - Visão Geral Estrutural da Base

Nesta seção, avaliamos a estrutura inicial do dataset, incluindo:

- Dimensão da base (linhas e colunas)
- Tipos de dados
- Estrutura geral das variáveis

O objetivo é identificar inconsistências estruturais antes de avançar para análises de qualidade.

In [3]:
print(f"O conjunto de dados apresenta: {df.shape[0]} linhas e {df.shape[1]} colunas")

O conjunto de dados apresenta: 13094 linhas e 10 colunas


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13094 entries, 0 to 13093
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID da Venda         13094 non-null  int64  
 1   Data da Venda       13094 non-null  str    
 2   Produto             13094 non-null  str    
 3   Categoria           13094 non-null  str    
 4   Preço Unitário      13094 non-null  float64
 5   Quantidade Vendida  13094 non-null  int64  
 6   Desconto            13094 non-null  str    
 7   Valor Total         13094 non-null  float64
 8   Localização         13094 non-null  str    
 9   Forma de Pagamento  13094 non-null  str    
dtypes: float64(2), int64(2), str(6)
memory usage: 1023.1 KB


### 📋 Validação de Tipagem

| Variável               | Tipo Esperado     | Tipo Encontrado | Status |
|------------------------|------------------|-----------------|--------|
| ID da Venda            | int              | int64           | ✅ OK |
| Data da Venda          | datetime         | str             | ⚠ Ajustar |
| Produto                | string           | str             | ✅ OK |
| Categoria              | string           | str             | ✅ OK |
| Preço Unitário         | float            | float64         | ✅ OK |
| Quantidade Vendida     | int              | int64           | ✅ OK |
| Desconto               | float            | str             | ⚠ Ajustar |
| Valor Total            | float            | float64         | ✅ OK |
| Localização            | string           | str             | ✅ OK |
| Forma de Pagamento     | string           | str             | ✅ OK |

---
**Observações:**

- A variável **Data da Venda** encontra-se como string e deverá ser convertida para datetime para permitir análises temporais adequadas.
- A variável **Desconto** está como string, indicando possível inconsistência de formatação ou mistura de tipos.
- As demais variáveis apresentam tipagem coerente com as regras de negócio.

---
# 04 - Avaliação de Qualidade dos Dados

## 04.1 Integridade Estrutural
Nesta etapa, avaliamos a integridade da base sob a perspectiva de qualidade, incluindo:

- Valores ausentes
- Registros duplicados
- ID inconsistente
- Desconto inconsistente
- Valor Total incorreto

### 04.1.1 - Valores Ausentes

In [5]:
df.isnull().sum()

ID da Venda           0
Data da Venda         0
Produto               0
Categoria             0
Preço Unitário        0
Quantidade Vendida    0
Desconto              0
Valor Total           0
Localização           0
Forma de Pagamento    0
dtype: int64

---
Não foram identificados valores nulos formais (NaN) na base.
Entretanto, a ausência de NaN não garante ausência de inconsistências.
Será necessário avaliar possíveis valores inválidos representados como strings.

### 04.1.2 - Registros Duplicados

In [6]:
df.duplicated().sum()

np.int64(0)

### 04.1.3 - ID inconsistente

In [7]:
df["ID da Venda"].duplicated().sum()

np.int64(130)

In [8]:
df.groupby("ID da Venda")["Produto"].count().sort_values(ascending=False).head()

ID da Venda
9484     131
1020       1
14079      1
14078      1
1003       1
Name: Produto, dtype: int64

In [9]:
df[df["ID da Venda"] == 9484]

,ID da Venda,Data da Venda,Produto,Categoria,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Localização,Forma de Pagamento
135,9484,2023-02-02 18:45:52.234823215,Calça E,Roupas,120.00,-1,0.07,223.20,Rio de Janeiro,Cartão de Crédito
142,9484,2023-02-04 11:33:34.943295530,Camisa D,Roupas,80.00,2,0.15,136.00,Curitiba,Pix
209,9484,2023-02-20 18:01:40.867244829,Calça E,Roupas,120.00,2,0.08,220.80,Belo Horizonte,Cartão de Crédito
223,9484,2023-02-24 03:37:06.284189459,Camisa D,Roupas,80.00,1,0.09,72.80,Belo Horizonte,Pix
232,9484,2023-02-26 08:04:09.766511007,Camisa D,Roupas,80.00,4,0.04,307.20,São Paulo,Pix
...,...,...,...,...,...,...,...,...,...,...
12421,9484,2025-11-25 01:16:15,Tênis E,Moda,338.84,3,0.07,945.36,Porto Alegre,Boleto
12500,9484,2025-11-29 08:33:04,Camisa V,Moda,301.91,5,18%,1237.83,Belo Horizonte,Pix
12568,9484,2025-12-02 11:52:46,Tênis L,Moda,340.77,4,0.08,1254.03,Salvador,Cartão credito
12637,9484,2025-12-05 15:30:13,Calça C,Moda,326.14,4,0.18,1069.74,Porto Alegre,Cartão de Débito


In [10]:
df[df["ID da Venda"] == 9484].nunique()

ID da Venda             1
Data da Venda         130
Produto                46
Categoria              11
Preço Unitário         46
Quantidade Vendida      7
Desconto               22
Valor Total           129
Localização            14
Forma de Pagamento      8
dtype: int64

---
**Integridade do Campo "ID da Venda"**

Foi identificado que o identificador apresenta reutilização ao longo do tempo.

Exemplo: o ID 9484 aparece 131 vezes com datas e atributos distintos.

Conclusão:

- O campo não pode ser considerado chave primária.
- Será necessário reavaliar seu uso na etapa de preparação de dados.
- Ajustes serão realizados no notebook 02_data_cleaning.

### 04.1.4 - Desconto Inconsistente

In [11]:
df["Desconto"].unique()[:30]

<StringArray>
['0.05', '0.04', '0.02', '0.07', '0.09',  '0.1', '0.06', '0.08', '0.01',
 '0.13', '0.03', '0.11',   '5%',  '0.2', '0.12', '0.14', '0.15', '0.16',
 '0.18', '0.17',   '3%',  '0.0',   '4%',   '8%',   '1%', '0.22',  '12%',
   '9%', '0.19',  '16%']
Length: 30, dtype: str

---
**Inconsistência na Coluna "Desconto"**

A variável apresenta mistura de formatos:

- Percentual em formato decimal (ex: 0.05)
- Percentual em formato textual (ex: "5%")

Isso inviabiliza cálculos diretos e pode gerar distorções no valor total.

A padronização será realizada na etapa de preparação de dados.

---
Apenas para efeito de auditoria, vamos padronizar a coluna Desconto termporariamente para testes.

In [12]:
df_desc = df.copy()

df_desc["Desconto_temp"] = df_desc["Desconto"].str.replace("%", "", regex=False).astype(float)
df_desc.loc[df_desc["Desconto_temp"] > 1, "Desconto_temp"] = (df_desc["Desconto_temp"] / 100)

### 04.1.5 - Valor Total Incorreto

---
Validação da Regra de Negócio:
$$
Valor\ Total = Preço\ Unitário \times Quantidade\ Vendida \times (1 - Desconto)
$$

In [13]:
df_desc["Valor_Calculado"] = (
    df_desc["Preço Unitário"] * df_desc["Quantidade Vendida"] * 
    (1 - df_desc["Desconto_temp"])
)

df_desc["Diferença"] = df_desc["Valor Total"] - df_desc["Valor_Calculado"]

df_desc["Diferença"].abs().describe()

count    13094.000000
mean       157.474805
std       1108.789982
min          0.000000
25%          0.000600
50%          0.002000
75%          0.004000
max      16489.128500
Name: Diferença, dtype: float64

---
Validação da Regra de Negócio:
$$
Valor\ Total = Preço\ Unitário \times (1 - Desconto)
$$

In [14]:
df_desc["Valor_Calc2"] = (
    df_desc["Preço Unitário"] *
    (1 - df_desc["Desconto_temp"])
)
(df_desc["Valor Total"] - df_desc["Valor_Calc2"]).abs().describe()

count    13094.000000
mean      2366.731889
std       2879.491321
min          0.000000
25%        248.385000
50%       1146.904250
75%       3332.340000
max      20790.000000
dtype: float64

---
Validação da Regra de Negócio:
$$
Valor\ Total = Preço\ Unitário \times Quantidade\ Vendida
$$

In [15]:
df_desc["Valor_Calc3"] = (
    df_desc["Preço Unitário"] *
    df_desc["Quantidade Vendida"]
)
(df_desc["Valor Total"] - df_desc["Valor_Calc3"]).abs().describe()

count    13094.000000
mean       521.819203
std       1204.055048
min          0.000000
25%         55.460000
50%        183.930000
75%        546.910000
max      16633.770000
dtype: float64

---
Validação da Regra de Negócio:
$$
Valor\ Total = Preço\ Unitário\ com\ desconto
$$

In [16]:
df_desc["Valor_Calc3"] = (
    df_desc["Preço Unitário"] *
    (1 - df_desc["Desconto_temp"])
)
df_desc["Valor_Calc3"] = df_desc["Valor Total"] - df_desc["Valor_Calc3"]
df_desc["Valor_Calc3"].abs().describe()

count    13094.000000
mean      2366.731889
std       2879.491321
min          0.000000
25%        248.385000
50%       1146.904250
75%       3332.340000
max      20790.000000
Name: Valor_Calc3, dtype: float64

In [17]:
df_desc.query("`Quantidade Vendida` == 1")["Valor_Calc3"].abs().describe()

count    2419.000000
mean       12.587049
std       159.656869
min         0.000000
25%         0.000800
50%         0.002000
75%         0.003800
max      3165.380000
Name: Valor_Calc3, dtype: float64

In [18]:
df_desc.query("`Quantidade Vendida` > 1")["Valor_Calc3"].abs().describe()

count    10411.000000
mean      2910.602365
std       2933.159086
min          0.000000
25%        676.200800
50%       1780.860400
75%       4466.664000
max      20790.000000
Name: Valor_Calc3, dtype: float64

---
**Inconsistência na Variável "Valor Total"**

A validação matemática demonstrou que:

$$
Valor\ Total = Preço\ Unitário \times (1 - Desconto)
$$

A variável não considera a multiplicação pela quantidade vendida.

Impacto:

- Receita total da base está subestimada.
- Cálculos financeiros derivados estarão incorretos.
- O nome da variável não corresponde à sua semântica real.

A correção será realizada na etapa de preparação dos dados.

## 04.2 - Consistência de Domínio e Regras Operacionais
Nesta seção, validamos:

- Preço Unitário
- Quantidade
- Datas
- Categorias
- Forma de Pagamento
- Localização
- Produto

### 04.2.1 - Preço Unitário

Regra esperada:
$$
Preço\ Unitário > 0
$$

In [19]:
df.query("`Preço Unitário`<= 0")

,ID da Venda,Data da Venda,Produto,Categoria,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Localização,Forma de Pagamento
133,1134,2023-02-02 07:06:31.460973982,Calça E,Roupas,0.0,2,0.13,208.80,São Paulo,Pix
213,1214,2023-02-21 17:20:22.414943295,Livro G,Livros,0.0,1,0.06,56.40,Salvador,Pix
272,1273,2023-03-08 01:11:05.243495664,Curso Online H,Educação,0.0,4,0.05,1900.00,Rio de Janeiro,Cartão de Crédito
457,1458,2023-04-21 23:20:36.824549700,Curso Online H,Educação,0.0,2,0.05,950.00,Salvador,Cartão de Crédito
500,1501,2023-05-02 09:56:33.462308206,Livro G,Livros,0.0,5,0.03,291.00,Rio de Janeiro,Cartão de Crédito
...,...,...,...,...,...,...,...,...,...,...
12434,13434,2025-11-25 16:00:48,Curso Online B,Educação,0.0,2,0.13,1229.64,Rio de Janeiro,Cartão de Débito
12507,13507,2025-11-29 14:45:55,Calça K,Moda,0.0,5,0.13,1548.56,São Paulo,Boleto
12573,13576,2025-12-02 12:55:37,Fone J,Eletrônicos,0.0,5,8%,12041.60,Recife,Cartão de Crédito
12797,13806,2025-12-14 07:24:32,Cafeteira J,Eletrodomésticos,0.0,4,0.13,3802.74,São Paulo,Cartão de Débito


In [20]:
131 / len(df)

0.01000458225141286

---
**Inconsistência em Preço Unitário**

Foram identificados 131 registros com Preço Unitário igual a zero, enquanto o Valor Total apresenta valores positivos significativos.

Isso indica inconsistência entre os campos monetários e compromete análises de receita baseadas no preço unitário.

Esse problema será tratado na etapa de preparação dos dados.

### 04.2.2 - Quantidade

In [21]:
df.query("`Quantidade Vendida` <= 0")

,ID da Venda,Data da Venda,Produto,Categoria,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Localização,Forma de Pagamento
23,1024,2023-01-06 14:02:28.899266177,Curso Online H,Educação,500.00,0,0.03,1940.00,Curitiba,Cartão de Crédito
30,1031,2023-01-08 06:50:11.607738492,Calça E,Roupas,120.00,-1,0.01,356.40,Curitiba,Cartão de Crédito
45,1046,2023-01-11 22:15:17.411607738,Camisa D,Roupas,80.00,-1,0.1,72.00,São Paulo,Cartão de Débito
118,1119,2023-01-29 15:41:25.657104736,Curso Online H,Educação,500.00,0,0.13,1305.00,Rio de Janeiro,Pix
119,1120,2023-01-29 21:31:06.044029353,Calça E,Roupas,120.00,0,0.14,103.20,Belo Horizonte,Cartão de Crédito
...,...,...,...,...,...,...,...,...,...,...
12971,13972,2025-12-24 13:07:38,Monitor R,Eletrônicos,2873.03,0,0.08,5286.38,Belo Horizonte,Cartão de Débito
13059,14054,2025-12-29 19:05:20,Cadeira C,Móveis,1791.17,0,0.08,8239.38,Florianópolis,Cartão de Débito
13070,14072,2025-12-30 14:17:10,Livro A,Livros,124.86,0,0.04,599.33,Belo Horizonte,Cartão de Débito
13087,14094,2025-12-31 12:43:28,Cafeteira L,Eletrodomésticos,1181.22,-1,0.13,5138.31,Florianópolis,Pix


---
**Inconsistência em "Quantidade Vendida"**

Foram identificados 264 registros (≈2% da base) com quantidade vendida menor ou igual a zero.

Esse comportamento pode indicar:

- Registros de cancelamento ou devolução
- Erros operacionais
- Ruído intencional na base

A ausência de uma variável de status impede a identificação precisa da causa.

Esse ponto será tratado na etapa de preparação de dados.

### 04.2.3 - Datas

In [22]:
df["Data da Venda"].min(), df["Data da Venda"].max()

('2023-01-01 00:00:00.000000000', '2026-01-15 10:00:00')

---
Apenas para efeito de auditoria, vamos padronizar a coluna Data termporariamente para testes.

In [23]:
df_data = df.copy()

df_data["Data_temp"] = pd.to_datetime(
    df_data["Data da Venda"],
    errors="coerce"
)

df_data["Data_temp"].min(), df_data["Data_temp"].max()


(Timestamp('2023-01-01 00:00:00'), Timestamp('2025-07-22 11:36:01'))

In [24]:
df_data["Data_temp"].isna().sum()

np.int64(3142)

In [25]:
df_data["Data_temp"].dt.year.value_counts().sort_index()

Data_temp
2023.0    1493
2024.0    4705
2025.0    3754
Name: count, dtype: int64

In [26]:
df_data["Data_temp"].dt.month.value_counts().sort_index()

Data_temp
1.0     1106
2.0      970
3.0     1126
4.0     1075
5.0     1159
6.0     1043
7.0      920
8.0      497
9.0      525
10.0     518
11.0     516
12.0     497
Name: count, dtype: int64

In [27]:
df_data[df_data["Data_temp"].isna()]["Data da Venda"].head(20)

630     2026-01-15 10:00:00
709     2026-01-15 10:00:00
749     2026-01-15 10:00:00
989     2026-01-15 10:00:00
1044    2026-01-15 10:00:00
1086    2026-01-15 10:00:00
1159    2026-01-15 10:00:00
1583    2026-01-15 10:00:00
1628    2026-01-15 10:00:00
1831    2026-01-15 10:00:00
1980    2026-01-15 10:00:00
2404    2026-01-15 10:00:00
2419    2026-01-15 10:00:00
2724    2026-01-15 10:00:00
2886    2026-01-15 10:00:00
3158    2026-01-15 10:00:00
3356    2026-01-15 10:00:00
3374    2026-01-15 10:00:00
3502    2026-01-15 10:00:00
3973    2026-01-15 10:00:00
Name: Data da Venda, dtype: str

In [28]:
df_data[df_data["Data_temp"].isna()]["Data da Venda"].unique()[:20]

<StringArray>
['2026-01-15 10:00:00', '2025-07-22 11:59:54', '2025-07-22 12:43:20',
 '2025-07-22 12:51:12', '2025-07-22 14:30:28', '2025-07-22 15:25:41',
 '2025-07-22 15:32:07', '2025-07-22 16:24:59', '2025-07-22 16:39:18',
 '2025-07-22 17:14:38', '2025-07-22 20:46:29', '2025-07-22 21:06:41',
 '2025-07-23 00:30:05', '2025-07-23 02:05:56', '2025-07-23 05:21:28',
 '2025-07-23 05:42:31', '2025-07-23 06:04:35', '2025-07-23 07:35:45',
 '2025-07-23 09:20:08', '2025-07-23 11:13:42']
Length: 20, dtype: str

In [29]:
df_data[df_data["Data_temp"].isna()]["Data da Venda"].str.len().describe()

count    3142.0
mean       19.0
std         0.0
min        19.0
25%        19.0
50%        19.0
75%        19.0
max        19.0
Name: Data da Venda, dtype: float64

In [30]:
df_data[df_data["Data_temp"].isna()]["Data da Venda"].head().apply(lambda x: repr(x))

630     '2026-01-15 10:00:00'
709     '2026-01-15 10:00:00'
749     '2026-01-15 10:00:00'
989     '2026-01-15 10:00:00'
1044    '2026-01-15 10:00:00'
Name: Data da Venda, dtype: str

In [31]:
df[df["Data da Venda"].str.contains("2026")].shape

(65, 10)

In [32]:
df_data[df_data["Data_temp"].isna()]["Data da Venda"].str.contains("2026").sum()

np.int64(65)

In [33]:
df_data["Data da Venda"].str.contains("\.").sum()

<>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\brudi\AppData\Local\Temp\ipykernel_23980\2316364192.py:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df_data["Data da Venda"].str.contains("\.").sum()


np.int64(9952)

---
**Inconsistência de Precisão Temporal**

A coluna apresenta dois padrões distintos:

- Timestamp com precisão de nanosegundos
- Timestamp com precisão de segundos

A conversão automática para datetime resultou em 3.142 valores não convertidos (≈24% da base).

O problema não está na validade das datas, mas na inconsistência de formatação.

A padronização será realizada na etapa de preparação dos dados.

### 04.2.4 - Categoria

In [34]:
df["Categoria"].value_counts()

Categoria
Eletrônicos          3754
Moda                 3200
Eletrodomésticos     1640
Móveis               1637
Livros               1009
Educação             1009
Roupas                583
Eletrônicos            71
Moda                   69
Eletrodomésticos       33
Educação               23
Livros                 23
Móveis                 23
Roupas                 20
Name: count, dtype: int64

In [35]:
df["Categoria"].apply(lambda x: repr(x)).unique()

<StringArray>
[           ''Livros'',            ''Roupas'',       ''Eletrônicos'',
          ''Educação'',           ''Roupas '',         ''Educação '',
            ''Móveis'',  ''Eletrodomésticos'',      ''Eletrônicos '',
              ''Moda'',           ''Livros '',             ''Moda '',
           ''Móveis '', ''Eletrodomésticos '']
Length: 14, dtype: str

In [36]:
df["Categoria"].value_counts().sum()

np.int64(13094)

In [37]:
df["Categoria"].str.strip().value_counts()

Categoria
Eletrônicos         3825
Moda                3269
Eletrodomésticos    1673
Móveis              1660
Livros              1032
Educação            1032
Roupas               603
Name: count, dtype: int64

In [38]:
df["Categoria"].nunique()

14

In [39]:
df["Categoria"].str.strip().nunique()

7

---
**Inconsistência na Variável "Categoria"**

A análise identificou 14 categorias distintas na base original.

Após remoção de whitespace terminal, o número real de categorias reduz para 7.

Conclusão:

- A duplicação era causada por espaços no final das strings.
- Esse problema fragmenta agregações e distorce análises por categoria.
- Será corrigido na etapa de preparação dos dados.

### 04.2.5 - Forma de Pagamento

In [40]:
df["Forma de Pagamento"].value_counts()

Forma de Pagamento
Cartão de Crédito    3470
Pix                  3231
Cartão de Débito     3180
Boleto               2951
Cartao de Credito      62
Credito                48
boleto                 43
Cartão credito         38
pix                    36
debito                 35
Name: count, dtype: int64

In [41]:
df["Forma de Pagamento"].nunique()

10

In [42]:
df["Forma de Pagamento"].str.strip().str.lower().value_counts()

Forma de Pagamento
cartão de crédito    3470
pix                  3267
cartão de débito     3180
boleto               2994
cartao de credito      62
credito                48
cartão credito         38
debito                 35
Name: count, dtype: int64

---
**Consolidação da Variável "Forma de Pagamento"**

Após normalização básica (strip + lower), persistem variações semânticas:

- "cartao de credito"
- "credito"
- "cartão credito"
- "debito"

Esses registros representam aproximadamente 1,4% da base.

Conclusão:

- O domínio real possui 4 categorias.
- Há inconsistência textual e semântica.
- Será necessária padronização via mapeamento na etapa de limpeza.

### 04.2.6 - Localização

In [43]:
df["Localização"].value_counts()

Localização
São Paulo           1462
Rio de Janeiro      1302
Belo Horizonte      1175
Salvador            1121
Curitiba            1120
Goiânia              958
Recife               955
Porto Alegre         954
Brasília             935
Campinas             925
Florianópolis        918
Fortaleza            909
curitiba              17
São P@ulo             17
rio de janeiro        15
SÃO PAULO             15
FORTALEZA             14
salvador              13
 Rio de Janeiro       12
 Curitiba             11
goiânia               11
RIO DE JANEIRO        10
C@mpinas              10
CURITIBA               9
belo horizonte         9
recife                 9
FLORIANÓPOLIS          9
Curitib@               9
Fort@leza              9
Rio de J@neiro         9
BELO HORIZONTE         7
Goiâni@                7
GOIÂNIA                7
Br@sília               7
são paulo              6
SALVADOR               6
 São Paulo             6
CAMPINAS               6
florianópolis          6
 Florianópoli

In [44]:
df["Localização"].nunique()

57

In [45]:
df["Localização"].str.strip().str.lower().nunique()

21

In [46]:
df["Localização"].str.strip().str.lower().str.replace("@", "a").nunique()

12

---
**Avaliação da Variável "Localização"**

Foram identificadas 57 categorias distintas na variável.

Após normalização básica (strip + lower), o número reduziu para 21.

Após correção de ruído textual (substituição de "@"), o domínio real apresentou 12 cidades válidas.

Conclusão:
A variável apresenta elevado nível de ruído textual, com fragmentação artificial de categorias.

Impacto:
Análises regionais seriam severamente distorcidas sem padronização prévia.

### 04.2.7 - Produto

In [47]:
df["Produto"].nunique()

49

In [48]:
df["Produto"].str.strip().str.lower().nunique()

49

In [49]:
df["Produto"].str.contains(r"[^a-zA-ZÀ-ÿ0-9\s]", regex=True).sum()

np.int64(0)

---
**Avaliação da Variável "Produto"**

A variável apresenta 49 categorias distintas.

Após normalização básica (strip + lower), o número de categorias permaneceu inalterado.

Não foram identificados:
- Variações de caixa
- Espaços invisíveis
- Caracteres especiais
- Fragmentação textual

Conclusão:
A variável apresenta domínio consistente e não requer tratamento estrutural.

## 04.3 - Conclusão da Auditoria de Qualidade dos Dados

## Resumo Executivo

A base de dados foi avaliada sob a perspectiva de integridade estrutural, consistência semântica e validação de regras de negócio.

### Principais Achados

**1. Identificadores**
- A variável "ID da Venda" apresenta duplicidades.
- O identificador não é único e não pode ser utilizado como chave primária sem tratamento.

**2. Tipagem Inconsistente**
- A variável "Desconto" apresenta formato misto (percentual como string e valor decimal).
- Necessária padronização para formato numérico.

**3. Fragmentação Categórica**
- "Forma de Pagamento" apresenta variações ortográficas e semânticas (~2% da base).
- "Localização" apresentou alta fragmentação (57 categorias aparentes para 12 cidades reais).

**4. Regras de Negócio**
- Foram identificados registros com:
  - Quantidade Vendida <= 0
  - Preço Unitário = 0
- Esses casos violam premissas comerciais básicas.

**5. Variáveis Consistentes**
- A variável "Produto" apresenta domínio íntegro e padronizado.

---

## Avaliação Geral da Qualidade

A base apresenta:

- Ruído textual significativo em variáveis categóricas
- Inconsistência de domínio em campos críticos
- Registros que violam regras básicas de negócio

Apesar disso, a estrutura geral do dataset é sólida e passível de tratamento.

---

## Próxima Etapa

Será iniciada a fase de:

**Preparação e Tratamento dos Dados**, incluindo:

- Padronização textual
- Correção de tipagem
- Consolidação categórica
- Definição de estratégia para registros inválidos

Somente após essa etapa a base estará apta para Análise Exploratória (EDA).